<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/DETR/DETR_panoptic_segmentation_minimal_example_(with_DetrFeatureExtractor).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DETR panoptic segmentation

In this notebook, we show that [DETR](https://huggingface.co/docs/transformers/model_doc/detr) can be extended to panoptic segmentation of an image by adding a mask head on top of the decoder.

`DetrFeatureExtractor` was renamed to `DetrImageProcessor`; this notebook uses the current API.

![DETR panoptic](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/detr_architecture.png)

## Set-up environment

In [ ]:
%pip install -q transformers torch timm pillow matplotlib requests

## Prepare an image using `DetrImageProcessor`

We retrieve an image on which we wish to test the model. Here, we use an image from the validation set of COCO.

In [ ]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000281759.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
image

We can prepare the image for the model using `DetrImageProcessor`. It will take care of resizing the image and normalizing the channels using the ImageNet mean and standard deviation.

In [ ]:
from transformers import DetrImageProcessor, DetrForSegmentation
import torch

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50-panoptic")
encoding = processor(image, return_tensors="pt")
print(encoding.keys())
print(encoding["pixel_values"].shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForSegmentation.from_pretrained("facebook/detr-resnet-50-panoptic")
model.to(device)
model.eval()

with torch.no_grad():
    outputs = model(**{k: v.to(device) for k, v in encoding.items()})

print(outputs.pred_masks.shape)

This returns a mask for each query (DETR uses 100 queries for COCO). Let us visualize the high-confidence ones:

In [ ]:
import math
import matplotlib.pyplot as plt

scores = outputs.logits.softmax(-1)[..., :-1].max(-1)[0]
keep = scores > 0.85
kept_masks = outputs.pred_masks[keep].detach().cpu().numpy()

ncols = 5
nrows = max(1, math.ceil(len(kept_masks) / ncols))
fig, axs = plt.subplots(ncols=ncols, nrows=nrows, figsize=(18, 10))
axs = axs.reshape(nrows, ncols)
for ax in axs.flat:
    ax.axis("off")
for i, mask in enumerate(kept_masks):
    ax = axs[i // ncols, i % ncols]
    ax.imshow(mask, cmap="cividis")
    ax.axis("off")
fig.tight_layout()

Finally, we merge the masks into a unified panoptic segmentation with `post_process_panoptic_segmentation`.

In [ ]:
width, height = image.size
result = processor.post_process_panoptic_segmentation(
    outputs, target_sizes=[(height, width)]
)[0]
print(result.keys())
print(result["segmentation"].shape)
print(len(result["segments_info"]))

Let's visualize the result:

In [ ]:
import numpy as np

segmentation = result["segmentation"].cpu().numpy()
colored = np.zeros((segmentation.shape[0], segmentation.shape[1], 3), dtype=np.uint8)
rng = np.random.default_rng(0)
max_id = int(segmentation.max())
for segment_id in range(max_id + 1):
    colored[segmentation == segment_id] = rng.integers(0, 255, size=3)

plt.figure(figsize=(15, 15))
plt.imshow(colored)
plt.axis("off")
plt.show()